In [0]:
# ===========================================
# Notebook Name:
# 02_build_deck_registry
#
# Purpose:
# Build a Gold registry for unique functional
# deck compositions identified by deck_hash.
#
# Input:
# pokemon.silver.decks
# pokemon.silver.tournament_results
# pokemon.silver.tournaments
#
# Output:
# pokemon.gold.deck_registry
#
# Grain:
# One row per deck_hash
# ===========================================

In [0]:
from pyspark.sql import functions as F

DECKS_TABLE = "pokemon.silver.decks"
TOURNAMENT_RESULTS_TABLE = (
    "pokemon.silver.tournament_results"
)
TOURNAMENTS_TABLE = "pokemon.silver.tournaments"

DECK_REGISTRY_TABLE = (
    "pokemon.gold.deck_registry"
)

print("Input :", DECKS_TABLE)
print("Input :", TOURNAMENT_RESULTS_TABLE)
print("Input :", TOURNAMENTS_TABLE)
print("Output:", DECK_REGISTRY_TABLE)

In [0]:
decks_df = spark.table(DECKS_TABLE)

tournament_results_df = spark.table(
    TOURNAMENT_RESULTS_TABLE
)

tournaments_df = spark.table(
    TOURNAMENTS_TABLE
)

print("Deck rows:", decks_df.count())
print(
    "Tournament result rows:",
    tournament_results_df.count(),
)
print(
    "Tournament rows:",
    tournaments_df.count(),
)

In [0]:
display(
    tournaments_df.select(
        "tournament_id",
        "event_title",
        "event_date_display",
        "source_scraped_at",
    )
)

In [0]:
tournament_dates_df = (
    tournaments_df
    .select(
        "tournament_id",
        "event_title",
        "event_date_display",
        "source_scraped_at",
    )
    .withColumn(
        "event_date",
        F.coalesce(
            F.to_date(
                "event_date_display",
                "yyyy/MM/dd",
            ),
            F.to_date(
                "event_date_display",
                "yyyy-MM-dd",
            ),
            F.to_date(
                "event_date_display",
            ),
        ),
    )
)

display(tournament_dates_df)

In [0]:
tournament_dates_df = (
    tournament_dates_df
    .withColumn(
        "effective_event_date",
        F.coalesce(
            F.col("event_date"),
            F.to_date("source_scraped_at"),
        ),
    )
)

In [0]:
deck_appearances_df = (
    tournament_results_df
    .filter(F.col("deck_hash").isNotNull())
    .join(
        tournament_dates_df.select(
            "tournament_id",
            "effective_event_date",
            "event_title",
        ),
        on="tournament_id",
        how="left",
    )
    .select(
        "deck_hash",
        "deck_hash_version",
        "deck_code",
        "tournament_id",
        "event_title",
        "effective_event_date",
        "rank",
        "player_id",
        "player_name",
    )
)

display(
    deck_appearances_df.orderBy(
        "effective_event_date",
        "rank",
    )
)

In [0]:
deck_performance_df = (
    deck_appearances_df
    .groupBy(
        "deck_hash",
        "deck_hash_version",
    )
    .agg(
        F.min(
            "effective_event_date"
        ).alias("first_seen_date"),
        F.max(
            "effective_event_date"
        ).alias("last_seen_date"),
        F.countDistinct(
            "tournament_id"
        ).alias("tournament_count"),
        F.count("*").alias(
            "result_appearance_count"
        ),
        F.countDistinct(
            "deck_code"
        ).alias("official_deck_code_count"),
        F.countDistinct(
            "player_id"
        ).alias("unique_player_count"),
        F.min("rank").alias("best_rank"),
        F.round(
            F.avg("rank"),
            2,
        ).alias("average_rank"),
        F.expr(
            "percentile_approx(rank, 0.5)"
        ).cast("int").alias("median_rank"),
        F.sum(
            F.when(
                F.col("rank") == 1,
                1,
            ).otherwise(0)
        ).alias("win_count"),
        F.sum(
            F.when(
                F.col("rank") <= 4,
                1,
            ).otherwise(0)
        ).alias("top4_count"),
        F.sum(
            F.when(
                F.col("rank") <= 8,
                1,
            ).otherwise(0)
        ).alias("top8_count"),
    )
)

In [0]:
deck_performance_df = (
    deck_performance_df
    .withColumn(
        "win_rate",
        F.col("win_count")
        / F.col("result_appearance_count"),
    )
    .withColumn(
        "top4_rate",
        F.col("top4_count")
        / F.col("result_appearance_count"),
    )
    .withColumn(
        "top8_rate",
        F.col("top8_count")
        / F.col("result_appearance_count"),
    )
    .withColumn(
        "win_rate_pct",
        F.round(F.col("win_rate") * 100, 2),
    )
    .withColumn(
        "top4_rate_pct",
        F.round(F.col("top4_rate") * 100, 2),
    )
    .withColumn(
        "top8_rate_pct",
        F.round(F.col("top8_rate") * 100, 2),
    )
)

In [0]:
deck_identity_df = (
    decks_df
    .groupBy(
        "deck_hash",
        "deck_hash_version",
    )
    .agg(
        F.min("deck_code").alias(
            "representative_deck_code"
        ),
        F.sort_array(
            F.collect_set("deck_code")
        ).alias("deck_codes"),
        F.max("total_cards").alias(
            "total_cards"
        ),
        F.max("unique_card_names").alias(
            "unique_card_names"
        ),
        F.max("card_type_rows").alias(
            "card_type_rows"
        ),
        F.min(
            "source_scraped_at"
        ).alias("first_scraped_at"),
        F.max(
            "source_scraped_at"
        ).alias("last_scraped_at"),
        F.max("canonical_string").alias(
            "canonical_string"
        ),
        F.min("is_valid").alias("is_valid"),
    )
)

In [0]:
deck_registry_df = (
    deck_identity_df
    .join(
        deck_performance_df,
        on=[
            "deck_hash",
            "deck_hash_version",
        ],
        how="left",
    )
    .fillna({
        "tournament_count": 0,
        "result_appearance_count": 0,
        "official_deck_code_count": 0,
        "unique_player_count": 0,
        "win_count": 0,
        "top4_count": 0,
        "top8_count": 0,
        "win_rate": 0.0,
        "top4_rate": 0.0,
        "top8_rate": 0.0,
        "win_rate_pct": 0.0,
        "top4_rate_pct": 0.0,
        "top8_rate_pct": 0.0,
    })
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

In [0]:
display(
    deck_registry_df.orderBy(
        F.col("best_rank").asc_nulls_last(),
        F.col(
            "result_appearance_count"
        ).desc(),
    )
)

In [0]:
duplicate_hash_df = (
    deck_registry_df
    .groupBy("deck_hash")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_hash_count = (
    duplicate_hash_df.count()
)

if duplicate_hash_count > 0:
    display(duplicate_hash_df)

    raise ValueError(
        f"{duplicate_hash_count} duplicated "
        "deck_hash rows detected"
    )

print("Validation passed: one row per deck_hash")

In [0]:
invalid_decks_df = (
    deck_registry_df
    .filter(
        (F.col("total_cards") != 60)
        | (~F.col("is_valid"))
    )
)

invalid_deck_count = invalid_decks_df.count()

if invalid_deck_count > 0:
    display(invalid_decks_df)

    raise ValueError(
        f"{invalid_deck_count} invalid decks detected"
    )

print("Validation passed: all decks contain 60 cards")

In [0]:
invalid_rates_df = (
    deck_registry_df
    .filter(
        (F.col("win_rate") < 0)
        | (F.col("win_rate") > 1)
        | (F.col("top4_rate") < 0)
        | (F.col("top4_rate") > 1)
        | (F.col("top8_rate") < 0)
        | (F.col("top8_rate") > 1)
    )
)

if invalid_rates_df.count() > 0:
    display(invalid_rates_df)

    raise ValueError(
        "Invalid performance rates detected"
    )

print("Validation passed: performance rates are valid")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DECK_REGISTRY_TABLE}
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    representative_deck_code STRING,
    deck_codes ARRAY<STRING>,
    total_cards INT,
    unique_card_names BIGINT,
    card_type_rows BIGINT,
    first_scraped_at TIMESTAMP,
    last_scraped_at TIMESTAMP,
    canonical_string STRING,
    is_valid BOOLEAN,
    first_seen_date DATE,
    last_seen_date DATE,
    tournament_count BIGINT,
    result_appearance_count BIGINT,
    official_deck_code_count BIGINT,
    unique_player_count BIGINT,
    best_rank INT,
    average_rank DOUBLE,
    median_rank INT,
    win_count BIGINT,
    top4_count BIGINT,
    top8_count BIGINT,
    win_rate DOUBLE,
    top4_rate DOUBLE,
    top8_rate DOUBLE,
    win_rate_pct DOUBLE,
    top4_rate_pct DOUBLE,
    top8_rate_pct DOUBLE,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Gold registry of unique functional Pokemon deck compositions'
""")

In [0]:
spark.sql(
    f"TRUNCATE TABLE {DECK_REGISTRY_TABLE}"
)

(
    deck_registry_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(DECK_REGISTRY_TABLE)
)

print("Gold deck registry rebuilt.")

In [0]:
display(
    spark.table(DECK_REGISTRY_TABLE)
    .select(
        "deck_hash",
        "representative_deck_code",
        "deck_codes",
        "first_seen_date",
        "last_seen_date",
        "tournament_count",
        "result_appearance_count",
        "best_rank",
        "average_rank",
        "win_count",
        "top4_count",
        "top8_count",
    )
    .orderBy(
        F.col("best_rank").asc_nulls_last()
    )
)

In [0]:
%sql

SELECT
    representative_deck_code,
    deck_hash,
    tournament_count,
    result_appearance_count,
    best_rank,
    average_rank,
    win_count,
    top4_count,
    top8_count,
    first_seen_date,
    last_seen_date
FROM pokemon.gold.deck_registry
ORDER BY
    best_rank ASC,
    result_appearance_count DESC;